# 03 Training Linear Model Using Colab CPU Resources

# ============================================================
# Colab-ready tokenization demo (character → word → subword)
# Works with ~8 GB RAM easily
# ============================================================

In [2]:
!pip install -q tokenizers   # only needed once per runtime

import torch
import torch.nn.functional as F
import math
import os
from collections import Counter
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tokenizers.processors import TemplateProcessing

# ------------------------------------------------------------
# 1. Download data (Tiny Shakespeare – classic starter dataset)
# ------------------------------------------------------------

In [3]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")
print("First 300 chars:\n", text[:300])
print("-" * 60)

Dataset length: 1,115,394 characters
First 300 chars:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us
------------------------------------------------------------


# ------------------------------------------------------------
# 2. Character-level (exactly what you already have)
# ------------------------------------------------------------

In [4]:
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size_char = len(chars)

print(f"Character vocab size: {vocab_size_char}")
print("Some characters:", chars[:20])

# Encode whole text as character indices
char_data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
print(f"Character sequence length: {len(char_data):,}")

Character vocab size: 65
Some characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G']
Character sequence length: 1,115,394


# ------------------------------------------------------------
# 3. Word-level (simple split on whitespace + keep punctuation)
# ------------------------------------------------------------

In [5]:
# Very basic word tokenizer (good enough for learning)
import re
words = re.findall(r"\w+|[^\w\s]", text)   # words or single punctuation
word_counts = Counter(words)
# Keep only words that appear at least a few times (to keep vocab reasonable)
min_freq = 2
vocab_words = [w for w, c in word_counts.items() if c >= min_freq]
word_to_idx = {w: i for i, w in enumerate(vocab_words)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size_word = len(vocab_words)

print(f"\nWord vocab size (freq ≥ {min_freq}): {vocab_size_word}")
print("Most common words:", word_counts.most_common(10))

# Encode text as word indices (unknown words become a special <UNK>)
UNK = "<UNK>"
if UNK not in word_to_idx:
    word_to_idx[UNK] = vocab_size_word
    idx_to_word[vocab_size_word] = UNK
    vocab_size_word += 1

word_data = []
for w in words:
    word_data.append(word_to_idx.get(w, word_to_idx[UNK]))
word_data = torch.tensor(word_data, dtype=torch.long)
print(f"Word sequence length: {len(word_data):,}")


Word vocab size (freq ≥ 2): 7286
Most common words: [(',', 19846), (':', 10316), ('.', 7885), ("'", 6187), ('the', 5442), ('I', 5043), ('to', 4112), ('and', 3763), (';', 3628), ('of', 3314)]
Word sequence length: 262,927


# ------------------------------------------------------------
# 4. Subword-level (BPE – the modern standard)
# ------------------------------------------------------------

In [6]:
# Train a tiny BPE tokenizer on the same text
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()   # split on spaces first
trainer = trainers.BpeTrainer(
    vocab_size=1000,          # small for demo – real models use 8k–50k
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"]
)
tokenizer.train_from_iterator([text], trainer=trainer)

# Optional: make decoding nicer
tokenizer.decoder = decoders.BPEDecoder()

# Save so you can reload later
tokenizer.save("shakespeare_bpe.json")
print("\nBPE tokenizer trained and saved as shakespeare_bpe.json")

# Encode the whole text with subword tokens
encoded = tokenizer.encode(text)
subword_ids = torch.tensor(encoded.ids, dtype=torch.long)
vocab_size_sub = tokenizer.get_vocab_size()

print(f"Subword (BPE) vocab size: {vocab_size_sub}")
print(f"Subword sequence length: {len(subword_ids):,}")
print("Example encoding of first sentence:")
print(tokenizer.encode(text[:100]).tokens)


BPE tokenizer trained and saved as shakespeare_bpe.json
Subword (BPE) vocab size: 1000
Subword sequence length: 383,663
Example encoding of first sentence:
['First', 'Citizen', ':', 'Be', 'fore', 'we', 'pro', 'ce', 'ed', 'any', 'f', 'ur', 'ther', ',', 'hear', 'me', 'speak', '.', 'All', ':', 'S', 'pe', 'ak', ',', 'speak', '.', 'First', 'Citizen', ':', 'You']


# ------------------------------------------------------------
# 5. Simple bigram model that works with ANY of the three
# ------------------------------------------------------------

In [7]:
def build_bigram_probs(data, vocab_size):
    """Count consecutive pairs and turn into probabilities."""
    counts = torch.zeros((vocab_size, vocab_size), dtype=torch.float32)
    for i in range(len(data) - 1):
        counts[data[i], data[i + 1]] += 1
    # Normalize rows → probability distribution
    row_sums = counts.sum(dim=1, keepdim=True)
    probs = counts / row_sums.clamp(min=1e-8)   # avoid division by zero
    return probs

# Build for each level (you can comment out the ones you don’t need)
print("\nBuilding bigram probability matrices…")
char_probs   = build_bigram_probs(char_data,   vocab_size_char)
word_probs   = build_bigram_probs(word_data,   vocab_size_word)
subword_probs = build_bigram_probs(subword_ids, vocab_size_sub)
print("Done.")


Building bigram probability matrices…
Done.


# ------------------------------------------------------------
# 6. Sampling / generation helper
# ------------------------------------------------------------

In [8]:
def generate(probs, idx_to_token, start_token, length=200, temperature=1.0):
    """Generate a sequence by sampling from the bigram table."""
    # Convert start token → index
    if isinstance(start_token, str):
        # try to find it
        start_idx = None
        for i, t in idx_to_token.items():
            if t == start_token:
                start_idx = i
                break
        if start_idx is None:
            start_idx = 0
    else:
        start_idx = start_token

    current = start_idx
    generated = [idx_to_token[current]]

    for _ in range(length - 1):
        p = probs[current]
        # optional temperature (softens / sharpens the distribution)
        if temperature != 1.0:
            p = p ** (1.0 / temperature)
            p = p / p.sum()
        next_idx = torch.multinomial(p, 1).item()
        generated.append(idx_to_token[next_idx])
        current = next_idx
    return generated

# ------------------------------------------------------------
# 7. Generate examples
# ------------------------------------------------------------

In [9]:
print("\n" + "="*60)
print("CHARACTER-LEVEL sample:")
print("".join(generate(char_probs, idx_to_char, start_token="A", length=200)))

print("\n" + "="*60)
print("WORD-LEVEL sample:")
print(" ".join(generate(word_probs, idx_to_word, start_token="The", length=50)))

print("\n" + "="*60)
print("SUBWORD-LEVEL (BPE) sample:")
# For BPE we use the tokenizer’s id_to_token
id_to_token = {v: k for k, v in tokenizer.get_vocab().items()}
tokens = generate(subword_probs, id_to_token, start_token="The", length=80)
print(tokenizer.decode([tokenizer.token_to_id(t) for t in tokens if t in tokenizer.get_vocab()]))


CHARACTER-LEVEL sample:
An lor. tayiney mat leen, bitty ad tcee s grabengen h ou use; htnt s hinng M:
TEatomeadathe.
Tis grerey ica;

Sothexpimppaghesinethe w, tcolan o hee mas nde thecancuronor s l
I'l ggl h f mmaleedinl al

WORD-LEVEL sample:
The breath to depose The which his <UNK> the skin of York and cried , hearing how she within being Richard , had deserved no more . DUKE VINCENTIO : Well met , Which , and with trouble thee ; And , or two worthy man live in any thing

SUBWORD-LEVEL (BPE) sample:
Thedonotanyunderstandmelionta,come.Corwherebloodisbut'llbe,Iwerenoneelsebetheawear--ROMEO:Whatwasantigardofnatureisbemercy:ifhellundertakeherdwell,andafterlookuponhimthatsheep,arethycompletheethethandwithnoblance.
